<font color=#009afa size=50>DAT5001 - Visualisation of Data</font>

<font size=50>1 - Cleaning Satellite Dataset for Presentation</font>

|**Assessment Code:** |PRES1|
|---|---|
|**Author:**|Tom Rowan, ST20285213|
|**Course:**| DAT5001_S2_25|
|**Course Tutor:**|Dr. Imtiaz Khan|


## Initialise Notebook

In [1]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Import Python Libraries

In [2]:
# Standard Libraries
import json
import random
from datetime import datetime
import sys

# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.pyplot import figure
%matplotlib inline

# Seaborn Graphs
import seaborn as sns

# Cartopy Maps
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Geopandas
import geopandas as gpd

# GeoPy
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Pycountry 
import pycountry

# Tabulate
from tabulate import tabulate

# Colorama
from colorama import Fore, Back, Style

# Requests
import requests

# Beautifulsoup
from bs4 import BeautifulSoup

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Check for Internet Access

In [3]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

In [4]:
# Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.

# Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online


_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.planet4589.org',
                  'https://datascience.daemonchild.com']

if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    sys.exit('Stopping notebook running here.')


[INTERNET] Internet Connectivity Detected. Using Internet resources.


## Load the Datasets from the Internet

Reference Website:
https://www.planet4589.org/space/gcat/index.html

Tab Separated Value Files: 

- SatCat: https://www.planet4589.org/space/gcat/tsv/cat/satcat.tsv
- Events: https://planet4589.org/space/gcat/tsv/cat/ecat.tsv
- Payload: https://www.planet4589.org/space/gcat/tsv/cat/psatcat.tsv   
- Org Data: https://www.planet4589.org/space/gcat/tsv/tables/orgs.tsv
- Launch sites: https://www.planet4589.org/space/gcat/tsv/tables/sites.tsv

Documentation: https://www.planet4589.org/space/gcat/pdf/doc.pdf

Reference: "McDowell, J, 2020: General Catalog of Artificial Space Objects, https://planet4589.org/space/gcat", First accessed 24/10/2025

> VIII.2.1.1 The Standard (S) Satellite Catalog
>
> Since the late 1950s the US government has maintained a catalog of satellites (often referred to as the SATCAT). Each
satellite is given a sequential number (1 was the rocket stage from the first Sputnik). These numbers are sometimes
referred to as SSN numbers.

In [5]:
# This will get the very latest dataset from John McDowells's page. This dataset is updated daily!
satcat_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/cat/satcat.tsv', sep='\t')
events_df = pd.read_csv('https://planet4589.org/space/gcat/tsv/cat/ecat.tsv', sep='\t')
pay_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/cat/psatcat.tsv', sep='\t')
orgs_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/tables/orgs.tsv', sep='\t')
lsite_df = pd.read_csv('https://www.planet4589.org/space/gcat/tsv/tables/sites.tsv', sep='\t')

datasets = {'satcat_df': satcat_df, 
            'events_df': events_df, 
            'pay_df': pay_df, 
            'orgs_df': orgs_df,
            'lsite_df': lsite_df
            }

Save a local copy

In [6]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
timestamp_formatted = today.strftime("%Y%m%d")

# Create filenames, timestamped
filepath = '../../datasets/'

for k,v in datasets.items():
    filename = f'{filepath}{k}_{timestamp_formatted}.csv'
    v.to_csv(filename, index=False)


### Explore these datasets

In [7]:
for k,v in datasets.items():
    print ("-----")
    print ("Dataset: " + k)
    print ("")
    print (v.columns)
    print ("")
    print (v.shape)

-----
Dataset: satcat_df

Index(['#JCAT', 'Satcat', 'Launch_Tag', 'Piece', 'Type', 'Name', 'PLName',
       'LDate', 'Parent', 'SDate', 'Primary', 'DDate', 'Status', 'Dest',
       'Owner', 'State', 'Manufacturer', 'Bus', 'Motor', 'Mass', 'MassFlag',
       'DryMass', 'DryFlag', 'TotMass', 'TotFlag', 'Length', 'LFlag',
       'Diameter', 'DFlag', 'Span', 'SpanFlag', 'Shape', 'ODate', 'Perigee',
       'PF', 'Apogee', 'AF', 'Inc', 'IF', 'OpOrbit', 'OQUAL', 'AltNames'],
      dtype='object')

(66650, 42)
-----
Dataset: events_df

Index(['#JCAT', 'Satcat', 'Launch_Tag', 'Piece', 'Type', 'Name', 'PLName',
       'LDate', 'Parent', 'SDate', 'Primary', 'DDate', 'Status', 'Dest',
       'Owner', 'State', 'Manufacturer', 'Bus', 'Motor', 'Mass', 'MassFlag',
       'DryMass', 'DryFlag', 'TotMass', 'TotFlag', 'Length', 'LFlag',
       'Diameter', 'DFlag', 'Span', 'SpanFlag', 'Shape', 'ODate', 'Perigee',
       'PF', 'Apogee', 'AF', 'Inc', 'IF', 'OpOrbit', 'OQUAL', 'AltNames'],
      dtype='object

## Data Cleaning
Remove columns that are not useful for the report.

**Satcat Database**

In [8]:
# Remove columns we can't use
columns_to_keep = ['Piece', 'Type', 'Name', 'LDate', 'DDate', 'Status', 'Owner', 'State', 'Manufacturer', 'Mass', 'Length', 'Diameter', 'Span', 'Shape', 'Perigee', 'Apogee']
sdf = satcat_df[columns_to_keep]

# Rename Columns for easier reading
sdf.rename(columns={'Piece':'CosparID', 'LDate':'Launch Date', 'DDate':'Decay Date', 'Owner':'Operator Code','Manufacturer':'Manufacturer Code'}, inplace=True)

The date fields are a little challenging. Some contain just dates (US format), some contain date and times, and some contain "-" or " -". These are fixed now because other calculations will rely on solid dates.

In [9]:
sdf['Launch Date'] = sdf['Launch Date'].replace("-", np.nan)
sdf['Decay Date'] = sdf['Decay Date'].replace("-", np.nan)
sdf['Decay Date'] = sdf['Decay Date'].replace(" -", np.nan)

# Strip trailing time values: keep only first 3 tokens (year, month, day)
sdf['Launch Date'] = (
    sdf['Launch Date']
    .str.split()        # split into parts
    .str[:3]            # keep first 3
    .str.join(" ")      # join back into a string
)

sdf['Decay Date'] = (
    sdf['Decay Date']
    .str.split()        # split into parts
    .str[:3]            # keep first 3# Replace "-" with NaN

    .str.join(" ")      # join back into a string
)

# There are some empty rows - such as where JCAT = S49051
sdf = sdf.dropna(subset=['Launch Date'])


# There are some dates that are questionable - remove the question marks
sdf['Decay Date'] = sdf['Decay Date'].str.replace('?', '', regex=False)

In [10]:
sdf.shape

(66586, 16)

**Events Database**

In [11]:
edf_columns_to_keep = ['Piece', 'LDate', 'DDate', 'Status', 'Perigee', 'Apogee']
edf = events_df[edf_columns_to_keep]

edf.rename(columns={'Piece':'CosparID', 'LDate':'Launch Date', 'DDate':'Decay Date'}, inplace=True)

In [12]:
edf['Decay Date'] = edf['Decay Date'].replace("-", np.nan)
edf['Decay Date'] = edf['Decay Date'].replace(" -", np.nan)

# Strip trailing time values: keep only first 3 tokens (year, month, day)
edf['Launch Date'] = (
    edf['Launch Date']
    .str.split()        # split into parts
    .str[:3]            # keep first 3
    .str.join(" ")      # join back into a string
)

edf['Decay Date'] = (
    edf['Decay Date']
    .str.split()        # split into parts
    .str[:3]            # keep first 3# Replace "-" with NaN

    .str.join(" ")      # join back into a string
)

# There are some dates that are questionable - remove the question marks
edf['Decay Date'] = edf['Decay Date'].str.replace('?', '', regex=False)

**Organisations Database**

In [13]:
# Create the CombinedName column using owner_EName if available, otherwise owner_Name
orgs_df['CombinedName'] = orgs_df['EName'].fillna(orgs_df['Name'])

# Remove columns we can't use
orgs_columns_to_keep = ['#Code', 'ShortName', 'CombinedName', 'Longitude', 'Latitude', 'Parent']
odf = orgs_df[orgs_columns_to_keep]

# Rename Columns for easier reading
odf.rename(columns={'#Code':'Org Code', 'ShortName':'Short Name', 'CombinedName':'Long Name'}, inplace=True)

Convert Latitude and Longitude to City and State values

In [ ]:

import time

# Set up the geocoder
geolocator = Nominatim(user_agent="Cardiff_Met_Project_Tom_Rowan")

# Define a function to extract city and state
def get_location_info(lat, lon):
    print (".", end="", flush=True)
    time.sleep(2)
    lat_lon = f'{lat},{lon}'
    
    try:
        location = geolocator.reverse(lat_lon, language="english")
        if location and location.raw.get('address'):
            address = location.raw['address']
            city = address.get('city', address.get('town', address.get('village', '')))
            state = address.get('state', '')

            return f"{city}|{state}|{country}|{country_code}"
    except:
        return None
    
#filtered_odf = odf.sample(5)    

# Apply the function using a lambda
odf['GeoInfo'] = odf.apply(lambda row: get_location_info(row['Latitude'], row['Longitude']), axis=1)

In [ ]:
odf[['City', 'State', 'Country', 'Country Code']] = odf['GeoInfo'].str.split('|', expand=True)
odf.to_csv('../../datasets/odf-complete-version.csv', index=False)
#odf = pd.read_csv('../../datasets/complete-version.csv')

In [53]:
odf.sample(20)

,Org Code,Short Name,Long Name,Longitude,Latitude,Parent,CityState
2002,LEID,Leiden,-,4.4900,52.1600,-,Leiden|Zuid-Holland
2035,LMEW,LM/EW,-,-74.5300,40.2600,LM,Hightstown|New Jersey
2233,MIT,MIT,-,-71.0900,42.3600,-,Cambridge|Massachusetts
340,ADSDF,Airbus/DF,-,9.4800,47.6500,ADS,Friedrichshafen|Baden-Württemberg
2684,OSCW,OSC Wallops,-,-75.4900,37.8400,OSC,|Virginia
2329,NAAA,NAA Aerophysics,-,-118.3500,33.9600,NAA,Inglewood|California
269,45SWF,USSF 45SW,-,-80.6100,28.2400,USAF,South Patrick Shores|Florida
949,CLEP,THGZ,China Lunar Exploration and Aerospace Engineer...,116.3800,39.9200,CNSA,西城区|北京市
1187,DYN,Dynacon,-,-79.6500,43.6000,-,Mississauga|Ontario
1163,DRTE,DRTE,-,-75.6800,45.4200,-,Ottawa|Ontario


**Payload Database**

In [17]:
payload_columns_to_keep = ['Piece', 'Program', 'Class', 'Category']
pdf = pay_df[payload_columns_to_keep]

# Rename 'Piece' to 'CosparID'
pdf.rename(columns={'Piece':'CosparID'}, inplace=True)

## Combine the Datasets

In [18]:
# Rename odf columns for Operator merge
owner_odf = odf.rename(columns=lambda x: f'Operator {x}' if x != 'Org Code' else 'Operator Code')
sdf = sdf.merge(owner_odf, on='Operator Code', how='left')

# Repeat for Manufacturer merge
manufacturer_odf = odf.rename(columns=lambda x: f'Manufacturer {x}' if x != 'Org Code' else 'Manufacturer Code')
sdf = sdf.merge(manufacturer_odf, on='Manufacturer Code', how='left')

# Merge payload data
sdf = sdf.merge(pdf, on='CosparID', how='left')

### Using Events to Update the Dataset

In [19]:
sdf['Decay Date'].isna().value_counts()

Decay Date
False    35266
True     31320
Name: count, dtype: int64

The events database contains updates to the Satcat data, so these need to be applied like a database log to get the updated values.

For example, this ID shows as being docked.

In [20]:
sdf[sdf['CosparID'] == '1988-038A']['Status']

19116    DK
Name: Status, dtype: object

However, later on in the events database, it undocked (UDK) and then finally deorbited (D):

In [21]:
edf[edf['CosparID'] == '1988-038A']['Status']

767    UDK
770      D
Name: Status, dtype: object

In [22]:
# Ensure types are comparible
sdf['Launch Date'] = pd.to_datetime(sdf['Launch Date'], errors='coerce')
sdf['Decay Date'] = pd.to_datetime(sdf['Decay Date'], errors='coerce')

edf['Launch Date'] = pd.to_datetime(edf['Launch Date'], errors='coerce')
edf['Decay Date'] = pd.to_datetime(edf['Decay Date'], errors='coerce')

# Dedupe edf to the latest update per CosparID by index, therefore keeping the last update.
edf_latest = (
    edf.sort_index()                # ensure index order is respected
       .drop_duplicates('CosparID', keep='last')
)

# Merge and update only specified columns
merged = sdf.merge(edf_latest, on='CosparID', how='left', suffixes=('', '_update'))

# Columns to update from edf_latest
cols_to_update = ['Launch Date', 'Decay Date', 'Status', 'Perigee', 'Apogee']

for col in cols_to_update:
    update_col = f"{col}_update"
    if update_col in merged.columns:
        merged[col] = merged[update_col].combine_first(merged[col])

# Clean up _update columns
sdf = merged.drop(columns=[c for c in merged.columns if c.endswith('_update')])



Check with the example from above:

In [23]:
sdf[sdf['CosparID'] == '1988-038A']['Status']

19116    D
Name: Status, dtype: object

# Feature Engineering

### Category, Class and Type Fields

In [24]:
# These mappings were created from the documentation:

class_json = {
  "A": "Non-commercial",        # Academic / Amateur / Non-Profit
  "B": "Commercial",
  "C": "Government (Civil)",
  "D": "Government (Defence)",  # and Military and Intelligence
  "U": "Unknown"
}

type_json = {
  "P": "Payload",
  "C": "Component",
  "R": "Rocket Body",
  "D": "Debris",
  "S": "Suborbital payload",
  "X": "Deleted Object",
  "Z": "Spurious Entry"
}

category_json = {
  "AST": "Astronomy",
  "BIO": "Biology",
  "CAL": "Calibration",
  "COM": "Communications",
  "EDU": "Education",
  "EOSCI": "Earth observing science",
  "EW": "Missile early warning",
  "GEOD": "Geodesy",
  "IMG": "Imaging (optical)",
  "IMG-R": "Imaging (radar)",
  "INF": "Infrastructure",
  "MET": "Meteorology (imaging)",
  "MET-RO": "Meteorology (radio occultation)",
  "MGRAV": "Microgravity experiments",
  "MISC": "Miscellaneous",
  "NAV": "Navigation",
  "PLAN": "Deep space related mission",
  "RB": "Rocket body",
  "RV": "Reentry vehicle",
  "SCI": "Scientific studies",
  "SIG": "Signals intelligence",
  "SS": "Spaceship",
  "TARG": "Target",
  "TECH": "Technology",
  "WEAPON": "Weapon",
  "UNK": "Unknown"
}

status_json = {
    "O": "In orbit",
    "AO": "Attached in orbit",
    "AO IN": "Attached inside",
    "UDK": "Undocked",
    "REL": "Released",
    "DEP": "Deployed",
    "TO": "Takeoff Launch",
    "TOA": "Attached at takeoff",
    "OX": "In orbit",
    "E": "Exploded",
    "N": "Renamed",
    "NA": "Renamed attached",
    "LEASE": "Lease",
    "S": "Targeted suborbital entry/impact",
    "R": "Reentered",
    "D": "Deorbited",
    "L": "Landed",
    "LF": "Failed landing",
    "AS": "Attached Suborbital reentered",
    "AR": "Reentered attached",
    "AR IN": "Reentered inside",
    "AL": "Landed attached",
    "AL IN": "Landed inside",
    "TX": "Transmission ended",
    "F": "Failed",
    "AF": "Failed attached",
    "DK": "Docked",
    "GRP": "Grappled",
    "ATT": "Attached",
    "TFR": "Transfer Inside",
    "TFR E": "Transfer external Inside",
    "TFR IN": "Transfer in"
}

In [25]:
# Split the Category column at the first '/'
split_cols = sdf['Category'].str.split('/', n=1, expand=True)

# Remove trailing spurious characters from each part using a regex
sdf['Category'] = sdf['Category'].fillna('UNK')
sdf['Primary Category'] = split_cols[0].str.strip().str.replace(r'[*?]+$', '', regex=True) if split_cols.shape[0] > 1 else 'UNK'
sdf['Secondary Category'] = split_cols[1].str.strip().str.replace(r'[*?]+$', '', regex=True) if split_cols.shape[1] > 1 else 'UNK'

# Type field - Remove Sub-Types
sdf['Type'] = sdf['Type'].str[0]

# Remove X and Z objects ("Spurious and unknown")
sdf = sdf[~sdf['Type'].isin(['X', 'Z'])]

# Set Class to Unknown if not labelled
sdf['Class'] = sdf['Class'].fillna('U')

# Map the descriptions for better data visualisation labelling
sdf['Class Desc'] = sdf['Class'].map(class_json)
sdf['Type Desc'] = sdf['Type'].map(type_json)
sdf['Primary Category Desc'] = sdf['Primary Category'].map(category_json)
sdf['Secondary Category Desc'] = sdf['Secondary Category'].map(category_json)
sdf['Status Desc'] = sdf['Status'].map(status_json)

Delete coded columns and then swap round so that "Type Desc" becomes just "Type"

In [26]:
sdf.drop(columns=['Type', 'Class', 'Category', 'Primary Category', 'Secondary Category', 'Status'], inplace=True, axis=1)
sdf.rename(columns={'Type Desc':'Type'}, inplace=True)

# Rename columns ending with " Desc"
sdf = sdf.rename(
    columns=lambda c: c[:-5] if c.endswith(" Desc") else c
)

In [27]:
sdf['Type'].value_counts()

Type
Debris         27825
Payload        24635
Component       7905
Rocket Body     6195
Name: count, dtype: int64

In [28]:
sdf['Class'].value_counts()

Class
Unknown                 41837
Commercial              15246
Government (Defence)     5420
Government (Civil)       3154
Non-commercial            883
Name: count, dtype: int64

In [29]:
sdf['Primary Category'].value_counts()

Primary Category
Communications                     15097
Imaging (optical)                   2998
Technology                          1570
Calibration                          811
Spaceship                            765
Signals intelligence                 701
Navigation                           623
Scientific studies                   521
Meteorology (imaging)                303
Imaging (radar)                      268
Deep space related mission           257
Astronomy                            218
Missile early warning                213
Earth observing science              117
Geodesy                               65
Meteorology (radio occultation)       63
Weapon                                37
Microgravity experiments              31
Biology                               30
Target                                21
Reentry vehicle                       10
Education                              4
Name: count, dtype: int64

In [30]:
sdf['Status'].value_counts()

Status
Reentered             32218
In orbit              31634
Landed                 1326
Deorbited               401
Reentered attached      300
Attached in orbit        86
Landed attached          24
Lease                    14
Exploded                 12
Landed inside             3
Transfer Inside           3
Released                  2
Reentered inside          1
Renamed                   1
Name: count, dtype: int64

### Is the object still in orbit?

Work out whether the object is still in orbit by combining specific status wording and whether there is a Decayed Date recorded or not.

In [31]:
# Define category lists
not_in_orbit_categories = ['Reentered', 'Landed', 'Deorbited', 'Reentered attached', 'Reentered inside', 'Landed attached', 'Landed inside']
in_orbit_categories = ['In Orbit', 'Attached in orbit', 'Docked']

# If not in any of these categories, but has a decay date
sdf['In Orbit'] = (
    ( ~sdf['Status'].isin(not_in_orbit_categories) & ~sdf['Status'].isin(in_orbit_categories) & sdf['Decay Date'].notna() )
)

# Create the new column 'In Orbit'
sdf['In Orbit'] = (
    sdf['Status'].isin(in_orbit_categories) |       # Found in "In Orbit category"
    (~sdf['Status'].isin(not_in_orbit_categories))  # Not found in "No In Orbit category"
)



Returning to the example from before:

In [32]:
sdf[sdf['CosparID'] == '1988-038A']['In Orbit']

19116    False
Name: In Orbit, dtype: bool

In [33]:
sdf.sample(20)[['CosparID','Type', 'Status', 'Launch Date', 'Decay Date','In Orbit']]

,CosparID,Type,Status,Launch Date,Decay Date,In Orbit
41474,1998-067HV,Payload,Reentered,2015-12-06,2017-01-03,False
31740,1999-025CEQ,Debris,In orbit,1999-05-10,NaT,True
20748,1990-038G,Payload,Reentered,1990-04-25,1990-09-20,False
32772,1982-029G,Debris,In orbit,1982-04-07,NaT,True
19733,1988-116A,Payload,Landed,1988-12-29,1989-02-11,False
45901,2011-037DX,Debris,In orbit,2011-07-18,NaT,True
31865,1992-047AR,Debris,Reentered,1992-07-30,2010-08-30,False
32577,2006-057AZ,Debris,Reentered,2006-12-14,2008-04-25,False
24779,1997-018B,Rocket Body,Reentered,1997-04-21,2002-05-20,False
14577,1983-034D,Payload,Reentered,1983-04-19,1984-04-01,False


In [34]:
sdf['In Orbit'].value_counts().reset_index()

,In Orbit,count
0,False,34273
1,True,32287


In [35]:
# Check the results looks sane
filtered_df = sdf[(sdf['Type'] == 'Payload')]
                  
pivot_table = filtered_df.pivot_table(
    index='Status',
    columns='In Orbit',
    aggfunc='size',
    fill_value=0
)

# Display the result
print(pivot_table)

In Orbit            False  True 
Status                          
Attached in orbit       0     85
Deorbited             362      0
Exploded                0      8
In orbit                0  16722
Landed               1325      0
Landed attached        24      0
Landed inside           3      0
Lease                   0     14
Reentered            5449      0
Reentered attached    298      0
Released                0      1
Renamed                 0      1
Transfer Inside         0      1


### Generate Additional Date Values

The dates that are available are:

- Launch Date (always available)
- Decay Date (may be NULL or '?')

These values are converted from strings in the base dataset into python datetime objects.
This allows for comparisons and for additional values ot be inferred from them.

The following date columns were created:

- Launch Year
- Launch Month (Numerical)
- Launch Month (Text)
- Launch Decade (1950's: is 1950-1959.)
- Decay Year
- Decay Month (Numerical)
- Decay Month (Text)
- Decay Decade 

A temporary value for 'Effective Decay Date' is either copied from 'Decay Date' or set to 'Today' if the object is "In Orbit".
This allows for lifetime values in days, months and years to be calculated.

Finally, the datetime objects are reverted to strings (in UK format).

In [36]:
# Modify date columns
# Convert to datetime format
sdf['Launch Date'] = pd.to_datetime(sdf['Launch Date'], format='%Y %b %d', errors='coerce')
sdf['Decay Date'] = pd.to_datetime(sdf['Decay Date'], format='%Y %b %d', errors='coerce')

# Create a Year of Launch and Decay
sdf['Launch Year'] = sdf['Launch Date'].dt.year.astype('Int64')
sdf['Decay Year'] = sdf['Decay Date'].dt.year.astype('Int64')
sdf['Launch Month No'] = sdf['Launch Date'].dt.month.astype('Int64')
sdf['Decay Month No'] = sdf['Decay Date'].dt.month.astype('Int64')
sdf['Launch Month'] = sdf['Launch Date'].dt.strftime('%B')
sdf['Decay Month'] = sdf['Decay Date'].dt.strftime('%B')
sdf['Launch Decade'] = ((sdf['Launch Year'] // 10) * 10)
sdf['Decay Decade'] = ((sdf['Decay Year'] // 10) * 10)

# Replace null Decay Date with today's date
#sdf['Effective Decay Date'] = sdf['Decay Date'].fillna(today)
# If Status == "In Orbit", set to today, else use Decay Date
sdf['Effective Decay Date'] = np.where(
    sdf['In Orbit'] == True,
    today,
    sdf['Decay Date']
)

# Convert to datetime 
sdf['Effective Decay Date'] = pd.to_datetime(sdf['Effective Decay Date'], errors='coerce')

# Calculate lifetime
delta = sdf['Effective Decay Date'] - sdf['Launch Date']
sdf['Lifespan Days'] = delta.dt.days
sdf['Lifespan Months'] = (delta.dt.days / 30.44).round(1)  # Approximate month length
sdf['Lifespan Years'] = (delta.dt.days / 365.25).round(2)  # Account for leap years

# Format as dd/mm/yyyy (because we are British)
sdf['Launch Date'] = sdf['Launch Date'].dt.strftime('%d/%m/%Y')
sdf['Decay Date'] = sdf['Decay Date'].dt.strftime('%d/%m/%Y')

# Drop Effective Decay Date column
sdf = sdf.drop('Effective Decay Date', axis=1)

In [37]:
sdf.sample(20)[['CosparID','Type', 'Status', 'Launch Date', 'Decay Date','Lifespan Days','In Orbit']]

,CosparID,Type,Status,Launch Date,Decay Date,Lifespan Days,In Orbit
4657,1970-025AS,Debris,In orbit,08/04/1970,NaN,20323.0,True
28774,2005-026A,Payload,Landed,26/07/2005,09/08/2005,14.0,False
23186,1994-044A,Payload,Landed,29/07/1994,11/09/1994,44.0,False
25460,1998-049B,Rocket Body,Reentered,25/08/1998,20/09/1998,26.0,False
16575,1979-017HK,Debris,Reentered,24/02/1979,08/08/1988,3453.0,False
11479,1977-097BB,Component,Reentered,29/09/1977,15/10/1979,746.0,False
27116,2001-049BM,Debris,Reentered,22/10/2001,13/01/2003,448.0,False
29033,1979-095L,Debris,Reentered,31/10/1979,19/05/2015,12984.0,False
48749,2002-032CK,Debris,Reentered,24/06/2002,27/09/2025,8496.0,False
5774,1972-003A,Payload,In orbit,23/01/1972,18/07/2001,19668.0,True


### Map Country Code to Country Name

The country codes ('State') supplied are two figure codes. 
These are first converted to three figure codes, then a Country name derived from this.

In [38]:
# Convert alpha-2 to alpha-3
def alpha2_to_alpha3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except:
        return None

# Apply conversion to your State column
sdf['Country Code'] = sdf['State'].apply(alpha2_to_alpha3)
sdf['Country'] = sdf['Country Code'].apply(lambda x: pycountry.countries.get(alpha_3=x).name if x else None)

# Drop State Column
sdf.drop(columns='State', axis=1, inplace=True)

In [39]:
# Create a combined "Program Root" field
#sdf["Program Root"] = sdf["Program"].str.split().str[0]
sdf["Program Root"] = sdf["Program"].str.extract(r"^([A-Za-z0-9]+)")

Tidy Apogee and Perigee columns

In [40]:
# Convert to numeric, non-numeric → NaN
sdf['Apogee'] = pd.to_numeric(sdf['Apogee'], errors='coerce')
sdf['Perigee'] = pd.to_numeric(sdf['Perigee'], errors='coerce')

# Flip negatives to positive
sdf['Apogee'] = sdf['Apogee'].abs()
sdf['Perigee'] = sdf['Perigee'].abs()

In [41]:
# Reorder columns alphabetically
sdf = sdf.reindex(sorted(sdf.columns), axis=1)

In [42]:
sdf.columns

Index(['Apogee', 'Class', 'CosparID', 'Country', 'Country Code', 'Decay Date',
       'Decay Decade', 'Decay Month', 'Decay Month No', 'Decay Year',
       'Diameter', 'In Orbit', 'Launch Date', 'Launch Decade', 'Launch Month',
       'Launch Month No', 'Launch Year', 'Length', 'Lifespan Days',
       'Lifespan Months', 'Lifespan Years', 'Manufacturer Code',
       'Manufacturer Latitude', 'Manufacturer Long Name',
       'Manufacturer Longitude', 'Manufacturer Parent',
       'Manufacturer Short Name', 'Mass', 'Name', 'Operator Code',
       'Operator Latitude', 'Operator Long Name', 'Operator Longitude',
       'Operator Parent', 'Operator Short Name', 'Perigee', 'Primary Category',
       'Program', 'Program Root', 'Secondary Category', 'Shape', 'Span',
       'Status', 'Type'],
      dtype='object')

In [43]:
sdf.shape

(66560, 44)

### Write out the cleaned dataset for further use



In [44]:
# Create filenames, one timestamped, one 'latest'
filename = f'{filepath}space_objects_{timestamp_formatted}.csv'
latestname = f'{filepath}space_objects_latest.csv'

# Write out CSV files
sdf.to_csv(filename, index=False)
sdf.to_csv(latestname, index=False)